# Data Exploratoration

Checks the shape, the values, and the overall usability of the data for this first ML project.

We first load the data in NumPy arrays using the provided helper function and then check their shape.

Recall that we use NumPy because Pandas is not authorizewd for this first project.

In [1]:
import numpy as np
from helpers import load_csv_data

x_train, x_test, y_train, train_ids, test_ids = load_csv_data("data/")

In [2]:
print("x_train:", x_train.shape)
print("x_test: ", x_test.shape)
print("y_train:", y_train.shape, "values:", np.unique(y_train))

x_train: (328135, 321)
x_test:  (109379, 321)
y_train: (328135,) values: [-1  1]


We then check what distribution of healthy vs unhealthy people are in the training dataset. 91.2% of people are healthy (−1) while only 8.8% have heart disease (+1).

This is probably one of the reason we are tested with F1 scores, as a model predicting that everyone is healthy would get overall 91% of correct prediction, while being completely incorrect.

In [3]:
def class_balance(y):
    values, counts = np.unique(y, return_counts=True)
    return dict(zip(values, counts / len(y)))


print(class_balance(y_train))

{np.int64(-1): np.float64(0.911697929205967), np.int64(1): np.float64(0.08830207079403295)}


Once these basics are done, we can start looking at what the data looks like.

We first print the "names" of the first 20 columns, and then the associated values for the first 10 rows.

We can already see how quite a few variables are not necessary, as the date the survey was done won't help us determine if the patient is sick or not. 

These are features that will have to be removed by hand I belive, as there is no automatic way to filter them.

In [ ]:
with open("data/x_train.csv") as f:
    feature_names = f.readline().strip().split(",")[1:]

print(len(feature_names))
print(feature_names[:20])

321
['_STATE', 'FMONTH', 'IDATE', 'IMONTH', 'IDAY', 'IYEAR', 'DISPCODE', 'SEQNO', '_PSU', 'CTELENUM', 'PVTRESD1', 'COLGHOUS', 'STATERES', 'CELLFON3', 'LADULT', 'NUMADULT', 'NUMMEN', 'NUMWOMEN', 'CTELNUM1', 'CELLFON2']


In [5]:
def show_features(x, rows=10, cols=20):
    """Print the first `rows` values of each of the first `cols` columns, one feature per line."""

    def fmt(v):
        if np.isnan(v):
            return "nan"
        return str(int(v)) if v == int(v) else f"{v:.2f}"

    for j in range(cols):
        vals = ", ".join(fmt(v) for v in x[:rows, j])
        print(f"col {j:3d}: {vals}")


show_features(x_train)

col   0: 53, 33, 20, 42, 24, 54, 39, 44, 21, 4
col   1: 11, 12, 10, 6, 11, 4, 12, 3, 7, 6
col   2: 11162015, 12152015, 10202015, 6182015, 11062015, 4082015, 12072015, 3072015, 7312015, 6232015
col   3: 11, 12, 10, 6, 11, 4, 12, 3, 7, 6
col   4: 16, 15, 20, 18, 6, 8, 7, 7, 31, 23
col   5: 2015, 2015, 2015, 2015, 2015, 2015, 2015, 2015, 2015, 2015
col   6: 1100, 1200, 1100, 1100, 1100, 1100, 1100, 1100, 1100, 1200
col   7: 2015015629, 2015004387, 2015005638, 2015004694, 2015004673, 2015004111, 2015005521, 2015000377, 2015007052, 2015006238
col   8: 2015015629, 2015004387, 2015005638, 2015004694, 2015004673, 2015004111, 2015005521, 2015000377, 2015007052, 2015006238
col   9: nan, 1, 1, nan, 1, nan, 1, 1, nan, nan
col  10: nan, 1, 1, nan, 1, nan, 1, 1, nan, nan
col  11: nan, nan, nan, nan, nan, nan, nan, nan, nan, nan
col  12: nan, 1, 1, nan, 1, nan, 1, 1, nan, nan
col  13: nan, 1, 2, nan, 1, nan, 1, 1, nan, nan
col  14: nan, nan, nan, nan, nan, nan, nan, nan, nan, nan
col  15: nan, 2, 2, 

Now that we know that we have some features that won't be useful, we should also check if there are features that are not relevant because of how their values look like.

The following cell enables to see the 30 poorest features. The 30th still misses more than 99% of it's columns, so it seems we will be able/need to remove a lot of the features before fitting any model.

The following cell gives an overview of the distribution of the types of feature. The almost empty 116 features should be removed first in my opinion. We can then look in more details at the one that survive that first filtering for non useful features.

In [6]:
def summarize_columns(x):
    """Per column: missing fraction, count of distinct non-missing values, min, max."""
    n_missing = np.isnan(x).mean(axis=0)
    n_unique = np.array(
        [np.unique(x[~np.isnan(x[:, j]), j]).size for j in range(x.shape[1])]
    )
    col_min = np.nanmin(x, axis=0)
    col_max = np.nanmax(x, axis=0)
    return n_missing, n_unique, col_min, col_max


def print_summary(n_missing, n_unique, col_min, col_max, n_show=30):
    order = np.argsort(-n_missing)  # most-missing columns first
    for j in order[:n_show]:
        print(
            f"col {j:3d} | {feature_names[j]} | missing {n_missing[j]:6.1%} | "
            f"unique {n_unique[j]:5d} | min {col_min[j]:9.1f} | max {col_max[j]:9.1f}"
        )


n_missing, n_unique, col_min, col_max = summarize_columns(x_train)
print_summary(n_missing, n_unique, col_min, col_max)

col  11 | COLGHOUS | missing 100.0% | unique     1 | min       1.0 | max       1.0
col  14 | LADULT | missing 100.0% | unique     2 | min       1.0 | max       2.0
col 191 | PCDMDECN | missing  99.9% | unique     8 | min       1.0 | max      21.0
col 148 | ASDRVIST | missing  99.9% | unique    11 | min       1.0 | max      98.0
col 147 | ASERVIST | missing  99.9% | unique     8 | min       1.0 | max      98.0
col 152 | ASNOSLEP | missing  99.9% | unique     7 | min       1.0 | max       8.0
col 153 | ASTHMED3 | missing  99.9% | unique     6 | min       1.0 | max       9.0
col 154 | ASINHALR | missing  99.9% | unique     9 | min       1.0 | max       9.0
col 151 | ASYMPTOM | missing  99.9% | unique     8 | min       1.0 | max       9.0
col 149 | ASRCHKUP | missing  99.9% | unique    18 | min       1.0 | max      99.0
col 150 | ASACTLIM | missing  99.9% | unique    25 | min       1.0 | max     999.0
col 146 | ASATTACK | missing  99.9% | unique     3 | min       1.0 | max       7.0
col 15

In [7]:
print("fully/near-empty (>80% missing):", (n_missing > 0.8).sum())
print("constant columns (unique == 1):  ", (n_unique == 1).sum())
print("look categorical (unique <= 10): ", (n_unique <= 10).sum())
print("look continuous (unique > 10):   ", (n_unique > 10).sum())

fully/near-empty (>80% missing): 116
constant columns (unique == 1):   6
look categorical (unique <= 10):  219
look continuous (unique > 10):    102


First filter : we filter all the columns which have more than 80% of empty inputs.

In [8]:
def drop_high_missing(x, names, max_missing=0.8):
    """Keep only columns whose missing fraction is <= max_missing.
    Returns the reduced matrix, the reduced names, and the kept-column mask."""
    missing_frac = np.isnan(x).mean(axis=0)
    keep = missing_frac <= max_missing
    return x[:, keep], [n for n, k in zip(names, keep) if k], keep

X_filtered, feature_names_filtered, kepted_mask = drop_high_missing(x_train, feature_names, max_missing=0.8)
print("X_filtered:", X_filtered.shape)
print("kepted_mask:", kepted_mask.sum())

X_filtered: (328135, 205)
kepted_mask: 205
